# Import modules

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

# Preparing metabolomics

In [ ]:
import pickle

# Save the list to a file
with open('y_train_features.pkl', 'rb') as file:
    y_features = pickle.load(file)

y_features = list(y_features)

In [ ]:
metabolite_names_train = pd.read_csv('y_train_tissue_name_map.csv')
metabolite_names_train = metabolite_names_train.dropna(subset = 'HMDB')
metabolite_names_train

In [ ]:
metabolite_names_fudan = pd.read_csv('Fudan_metabolite_name_map.csv')
metabolite_names_fudan = metabolite_names_fudan.dropna(subset = 'HMDB')
metabolite_names_fudan

In [ ]:
common_hmdb = list(set(metabolite_names_train['HMDB']).intersection(metabolite_names_fudan['HMDB']))
len(common_hmdb)

In [ ]:
metabolite_names_train = metabolite_names_train[metabolite_names_train['HMDB'].isin(common_hmdb)]
metabolite_names_train.drop_duplicates(subset = 'HMDB', inplace = True)
metabolite_names_train

In [ ]:
metabolite_names_train_query_hmdb_dict = metabolite_names_train[['Query', 'HMDB']].set_index('Query').to_dict()['HMDB']
metabolite_names_train_query_hmdb_dict

In [ ]:
with open('y_train_features_fudan_hmdb_common.pkl', 'wb') as file:
    pickle.dump(metabolite_names_train_query_hmdb_dict, file)

In [ ]:
metabolite_names_fudan = metabolite_names_fudan[metabolite_names_fudan['HMDB'].isin(common_hmdb)]
metabolite_names_fudan

In [ ]:
lipids = pd.read_csv('FUSCC_TNBC_LipidMets.csv', index_col = 1, skiprows = 1)
lipids = lipids.drop('Peak', axis = 'columns')
lipids.index.name = None
lipids

In [ ]:
lipids.columns = lipids.columns.str.replace(r'_T$', '', regex=True)

In [ ]:
lipids.head(3)

In [ ]:
polar_mets = pd.read_csv('FUSCC_TNBC_PolarMets.csv', index_col = 1, skiprows = 1)
polar_mets = polar_mets.drop('Peak', axis = 'columns')
polar_mets.index.name = None
polar_mets

In [ ]:
polar_mets.columns = polar_mets.columns.str.replace(r'_T$', '', regex=True)
polar_mets.head(3)

In [ ]:
metabolites = pd.concat([lipids, polar_mets], axis = 'index')
metabolites

In [ ]:
metabolites = metabolites.T
metabolites.head(3)

In [ ]:
desired_metabolites = metabolite_names_fudan['Query'].tolist()
metabolites = metabolites[desired_metabolites]
metabolites

In [ ]:
metabolite_names_fudan_query_hmdb_dict = metabolite_names_fudan[['Query', 'HMDB']].set_index('Query').to_dict()['HMDB']
metabolite_names_fudan_query_hmdb_dict

In [ ]:
metabolites_columns = []
for col in metabolites.columns:
  metabolites_columns.append(metabolite_names_fudan_query_hmdb_dict[col])
metabolites.columns = metabolites_columns
metabolites

In [ ]:
metabolites.to_csv('metabolites_fudan_hmdb.csv')

# Preparing RNA seq

In [ ]:
counts = pd.read_csv('mRNA_counts_hgncsymbol.txt', sep = '\t')
counts = counts.rename(columns = {'Unnamed: 0' : 'Symbol'})
counts

In [ ]:
counts.head(2)

In [ ]:
gene_names = pd.read_csv('ensembl_id_gene_name.csv')
gene_names = gene_names.rename(columns = {'gene_name' : 'Symbol'})
gene_names

In [ ]:
common_genes = set(gene_names.Symbol).intersection(set(counts.Symbol))

In [ ]:
len(common_genes)

In [ ]:
gene_names = gene_names[gene_names['Symbol'].isin(common_genes)]
gene_names = gene_names.drop_duplicates(subset = 'Symbol')
gene_names

In [ ]:
counts = counts[counts['Symbol'].isin(common_genes)]
counts

In [ ]:
gene_names = gene_names.set_index('Symbol')
gene_names

In [ ]:
counts = counts.drop_duplicates()
counts = counts.set_index('Symbol')
counts

In [ ]:
counts = pd.concat([gene_names, counts], axis = 'columns')
counts.head(3)

In [ ]:
counts = counts.set_index('gene_id')
counts.head(3)

# TPM normalization

In [ ]:
#TMM normalization with rnanorm
!pip install rnanorm
from rnanorm import TPM

In [ ]:
counts = counts.T
counts

In [ ]:
norm_counts = TPM(gtf = 'Homo_sapiens.GRCh38.113.gtf').fit(counts)
norm_counts = norm_counts.transform(counts)
norm_counts

In [ ]:
norm_counts = pd.DataFrame(norm_counts)
norm_counts.columns = counts.columns
norm_counts.index = counts.index
norm_counts

# Select genes in metabolism

In [ ]:
met_genes = pd.read_csv('human_gem_associated_genes.csv')
met_genes

In [ ]:
met_genes_select = met_genes[met_genes['gene_id'].isin(norm_counts.columns)]
met_genes_select = met_genes_select.gene_id.to_list()
len(met_genes_select)

In [ ]:
norm_counts = norm_counts[met_genes_select]
norm_counts

In [ ]:
norm_counts = norm_counts.T
norm_counts = pd.concat([met_genes.set_index('gene_id'), norm_counts], axis = 'columns')
norm_counts = norm_counts.set_index('Symbol')
norm_counts.index.name = None
norm_counts = norm_counts.T
norm_counts

In [ ]:
import joblib

model = joblib.load('ElasticNet.pkl')

In [ ]:
model_features = model.feature_names_in_
model_features

In [ ]:
filtered_data = norm_counts[model_features]

In [ ]:
filtered_data.to_csv('fudan_tpm_met_genes_y_features.csv')